# Project 4: Kernel Methods from Scratch

**Chapter:** 5 — Nonlinear Classification & Kernels

**Goal:** Handle non-linearly-separable data by mapping to higher dimensions. Implement polynomial and RBF kernels, see the kernel trick in action.

**What you'll build:**
1. Feature maps (explicit polynomial expansion)
2. Kernel functions (polynomial, RBF)
3. Kernelized perceptron
4. Visualization of nonlinear decision boundaries

**Rules:** No sklearn. NumPy + matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12

## 1. Non-linearly-separable Data

In [ ]:
def make_circles(n=200, noise=0.1, seed=42):
    """Inner ring = -1, outer ring = +1."""
    rng = np.random.default_rng(seed)
    n_half = n // 2
    # Outer ring
    theta_out = rng.uniform(0, 2*np.pi, n_half)
    r_out = 2.0 + rng.normal(scale=noise, size=n_half)
    X_out = np.column_stack([r_out * np.cos(theta_out), r_out * np.sin(theta_out)])
    # Inner ring
    theta_in = rng.uniform(0, 2*np.pi, n_half)
    r_in = 0.8 + rng.normal(scale=noise, size=n_half)
    X_in = np.column_stack([r_in * np.cos(theta_in), r_in * np.sin(theta_in)])
    
    X = np.vstack([X_out, X_in])
    y = np.array([1]*n_half + [-1]*n_half)
    idx = rng.permutation(n)
    return X[idx], y[idx]

def make_xor(n=200, seed=42):
    """XOR-like pattern — 4 clusters, diagonal pairs share a label."""
    rng = np.random.default_rng(seed)
    k = n // 4
    centers = [(1,1), (-1,-1), (1,-1), (-1,1)]
    labels = [1, 1, -1, -1]
    X = np.vstack([rng.normal(loc=c, scale=0.4, size=(k,2)) for c in centers])
    y = np.array([l for l in labels for _ in range(k)])
    idx = rng.permutation(len(y))
    return X[idx], y[idx]

def plot_data(X, y, ax=None, title=""):
    if ax is None: fig, ax = plt.subplots()
    ax.scatter(X[y==1,0], X[y==1,1], c='steelblue', marker='o', label='+1', edgecolors='k', linewidth=0.5, s=40)
    ax.scatter(X[y==-1,0], X[y==-1,1], c='salmon', marker='s', label='-1', edgecolors='k', linewidth=0.5, s=40)
    ax.set_title(title); ax.legend(); ax.set_aspect('equal')
    return ax

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
X_circ, y_circ = make_circles()
X_xor, y_xor = make_xor()
plot_data(X_circ, y_circ, ax1, "Circles (not linearly separable)")
plot_data(X_xor, y_xor, ax2, "XOR (not linearly separable)")
plt.tight_layout(); plt.show()

## 2. Feature Maps & Kernels — Your Turn

### The idea

If data isn't separable in the original space, map it to a higher-dimensional space where it is:

$$\phi(\mathbf{x}) = (x_1, x_2, x_1^2, x_2^2, x_1 x_2)$$

**Kernel trick:** Instead of computing $\phi(\mathbf{x})$ explicitly, compute $K(\mathbf{x}, \mathbf{x}') = \phi(\mathbf{x}) \cdot \phi(\mathbf{x}')$ directly.

**Common kernels:**
- Polynomial: $K(\mathbf{x}, \mathbf{x}') = (\mathbf{x} \cdot \mathbf{x}' + c)^d$
- RBF (Gaussian): $K(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$

In [ ]:
def polynomial_kernel(X1, X2, degree=2, c=1.0):
    """Compute polynomial kernel matrix K[i,j] = (x1_i · x2_j + c)^d
    
    Args:
        X1: (n1, d), X2: (n2, d)
    Returns:
        K: (n1, n2)
    """
    ### YOUR CODE ###
    raise NotImplementedError("(X1 @ X2.T + c) ** degree")


def rbf_kernel(X1, X2, gamma=1.0):
    """Compute RBF (Gaussian) kernel matrix K[i,j] = exp(-γ ||x1_i - x2_j||²)
    
    Hint: ||a-b||² = ||a||² + ||b||² - 2a·b
    """
    ### YOUR CODE ###
    raise NotImplementedError("Compute pairwise squared distances, then exp(-γ * dist)")

### Kernelized Perceptron

Instead of storing weights $\mathbf{w}$, store how many times each training point was a support vector (the dual representation).

In [ ]:
class KernelPerceptron:
    def __init__(self, kernel_fn, **kernel_params):
        self.kernel_fn = kernel_fn
        self.kernel_params = kernel_params
        self.alpha = None  # dual coefficients
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y, epochs=20):
        """Train the kernel perceptron.
        
        alpha[i] counts how many times point i was a mistake.
        Prediction: sign(Σ_i alpha[i] * y[i] * K(x_i, x))
        Update: if misclassified, alpha[i] += 1
        """
        n = len(y)
        self.X_train = X.copy()
        self.y_train = y.copy()
        self.alpha = np.zeros(n)
        
        K = self.kernel_fn(X, X, **self.kernel_params)
        
        history = []
        for ep in range(epochs):
            mistakes = 0
            for i in range(n):
                ### YOUR CODE ###
                # 1. Compute prediction: sign(Σ_j alpha[j] * y[j] * K[j, i])
                # 2. If wrong: alpha[i] += 1, mistakes += 1
                raise NotImplementedError("Kernel perceptron update")
            history.append(mistakes)
            if mistakes == 0:
                break
        return history
    
    def predict(self, X):
        """Predict on new data using stored support vectors."""
        K = self.kernel_fn(self.X_train, X, **self.kernel_params)
        scores = (self.alpha * self.y_train) @ K
        return np.sign(scores).astype(int)
    
    def accuracy(self, X, y):
        preds = self.predict(X)
        preds[preds == 0] = 1
        return np.mean(preds == y)

### Sanity checks

In [ ]:
# Test kernels
X_test = np.array([[1, 0], [0, 1]])
K_poly = polynomial_kernel(X_test, X_test, degree=2, c=1)
assert K_poly.shape == (2, 2)
assert abs(K_poly[0, 0] - 4.0) < 1e-6  # (1+1)^2 = 4
print("✓ Polynomial kernel correct")

K_rbf = rbf_kernel(X_test, X_test, gamma=1.0)
assert abs(K_rbf[0, 0] - 1.0) < 1e-6  # K(x,x) = 1
assert K_rbf[0, 1] < 1.0  # K(x,x') < 1 for x != x'
print("✓ RBF kernel correct")

# Test kernel perceptron on circles
kp = KernelPerceptron(rbf_kernel, gamma=1.0)
hist = kp.fit(X_circ, y_circ, epochs=20)
acc = kp.accuracy(X_circ, y_circ)
assert acc > 0.9, f"Kernel perceptron should get >90% on circles, got {acc:.1%}"
print(f"✓ Kernel perceptron accuracy on circles: {acc:.1%}")
print("\n🎉 All checks passed!")

## 3. Visualize Decision Boundaries

In [ ]:
def plot_kernel_boundary(model, X, y, title="", ax=None, resolution=100):
    if ax is None: fig, ax = plt.subplots()
    x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-1.5, 0, 1.5], colors=['#ffcccc', '#cce5ff'], alpha=0.4)
    ax.contour(xx, yy, Z, levels=[0], colors='k', linewidths=2)
    plot_data(X, y, ax=ax, title=title)
    return ax

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

configs = [
    (X_circ, y_circ, rbf_kernel, {'gamma': 1.0}, "Circles + RBF γ=1"),
    (X_circ, y_circ, polynomial_kernel, {'degree': 2, 'c': 1}, "Circles + Poly d=2"),
    (X_xor, y_xor, rbf_kernel, {'gamma': 2.0}, "XOR + RBF γ=2"),
    (X_xor, y_xor, polynomial_kernel, {'degree': 3, 'c': 1}, "XOR + Poly d=3"),
]

for (X, y, kfn, kp, title), ax in zip(configs, axes.flat):
    model = KernelPerceptron(kfn, **kp)
    model.fit(X, y, epochs=30)
    plot_kernel_boundary(model, X, y, title=f"{title} (acc={model.accuracy(X,y):.0%})", ax=ax)

plt.tight_layout(); plt.show()

## 4. Experiments

### A: Vary γ in the RBF kernel
Try γ = 0.1, 1, 5, 20. What do small vs large γ do to the boundary?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
for gamma, ax in zip([0.1, 1.0, 5.0, 20.0], axes):
    m = KernelPerceptron(rbf_kernel, gamma=gamma)
    m.fit(X_circ, y_circ, epochs=30)
    plot_kernel_boundary(m, X_circ, y_circ, title=f"γ={gamma}, acc={m.accuracy(X_circ, y_circ):.0%}", ax=ax)
plt.tight_layout(); plt.show()

**Your observation:** *(What happens with very small γ? Very large γ? Which overfits?)*

...

### B: Computational cost of the kernel matrix

The kernel trick avoids computing $\phi(\mathbf{x})$ explicitly, but computing the kernel matrix $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$ is $O(n^2 \cdot d)$ — quadratic in dataset size. Training the kernel perceptron then iterates over this matrix.

**Experiment:** Time both `rbf_kernel` computation and `KernelPerceptron.fit` for increasing dataset sizes. How does wall-clock time scale?

In [ ]:
import time

sizes = [50, 100, 200, 400, 800]
kernel_times = []
fit_times = []

for n in sizes:
    X_big, y_big = make_circles(n=n, seed=42)
    
    # Time kernel matrix computation
    t0 = time.perf_counter()
    K = rbf_kernel(X_big, X_big, gamma=1.0)
    kernel_times.append(time.perf_counter() - t0)
    
    # Time full fit (includes kernel computation + training iterations)
    t0 = time.perf_counter()
    kp_tmp = KernelPerceptron(rbf_kernel, gamma=1.0)
    kp_tmp.fit(X_big, y_big, epochs=10)
    fit_times.append(time.perf_counter() - t0)
    
    print(f"n={n:4d}  kernel: {kernel_times[-1]:.4f}s  fit: {fit_times[-1]:.4f}s")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.loglog(sizes, kernel_times, 'bo-', markersize=8, label='rbf_kernel')
ax1.loglog(sizes, fit_times, 'rs-', markersize=8, label='fit (10 epochs)')
# Reference O(n^2) line
ref = [kernel_times[0] * (n / sizes[0])**2 for n in sizes]
ax1.loglog(sizes, ref, 'k--', alpha=0.4, label='$O(n^2)$ reference')
ax1.set_xlabel('Dataset size (n)'); ax1.set_ylabel('Time (seconds)')
ax1.set_title('Kernel Methods: Wall-Clock Scaling')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Ratio plot: time / n^2 should be roughly constant
ax2.plot(sizes, [t / (n/100)**2 for t, n in zip(kernel_times, sizes)], 'bo-', label='kernel time / n²')
ax2.plot(sizes, [t / (n/100)**2 for t, n in zip(fit_times, sizes)], 'rs-', label='fit time / n²')
ax2.set_xlabel('Dataset size (n)'); ax2.set_ylabel('Time / n² (normalized)')
ax2.set_title('Is the scaling truly O(n²)?')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print("\nKey insight: kernel methods hit a wall as n grows. This is why")
print("approximate methods (random features, Nyström) matter for large datasets.")

## 5. What You Learned

1. Why can't a linear classifier solve the circles or XOR problem?
2. What does the kernel trick save you from computing?
3. What's the tradeoff with γ in the RBF kernel?

---
**Next project:** Neural Networks — learn the feature map instead of choosing it.